# sc-tumor-annotator — demo (v0.2)

Capability portrait, not a research result. Synthetic data only. This
notebook walks the pipeline and visualizes the v0.2 headline: an
expression-derived CNV score that separates malignant from normal epithelial
cells in a hard, subclonal-CNV regime.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sctumor.synth import generate_malignancy_cohort, STROMAL_TYPES
from sctumor import cnv, ablation

cohort = generate_malignancy_cohort(2100, seed=0)
print(cohort.n_cells, 'cells x', cohort.n_genes, 'genes')


## CNV inference and the malignant-cell score


In [ ]:
ref = np.isin(cohort.obs['cell_type'].to_numpy(), STROMAL_TYPES)
mat, ch = cnv.infer_cnv(cohort.expr, cohort.genes, ref)
score = cnv.cnv_score(mat, ch)
mal = cohort.obs['is_malignant'].to_numpy()

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(score[~mal], bins=40, alpha=0.6, label='normal', density=True)
ax.hist(score[mal], bins=40, alpha=0.6, label='malignant', density=True)
ax.set_xlabel('chromosome-length-normalized CNV score'); ax.set_ylabel('density')
ax.legend(); ax.set_title('CNV score separates malignant from normal'); fig.tight_layout()


In [ ]:
# CNV heatmap: epithelial cells sorted by malignancy, genome-ordered columns
epi = cohort.obs['compartment'].to_numpy() == 'epithelial'
order = np.argsort(mal[epi])
sub = mat[epi][order]
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(sub, aspect='auto', cmap='bwr', vmin=-0.6, vmax=0.6)
ax.set_xlabel('genome-ordered genes'); ax.set_ylabel('epithelial cells (sorted: normal -> malignant)')
ax.set_title('Inferred CNV track'); fig.colorbar(im, ax=ax, shrink=0.7); fig.tight_layout()


## CNV-feature ablation on the malignant call


In [ ]:
abl = ablation.cnv_ablation(cohort, n_splits=5)
labels = ['kNN\n(embedding)', 'CNV\nscalar only', 'embedding\n(30 PCs)', 'embedding\n+ CNV']
vals = [abl['knn_baseline_f1_mean'], abl['cnv_only_f1_mean'],
        abl['embedding_f1_mean'], abl['embedding_plus_cnv_f1_mean']]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(labels, vals)
ax.set_ylim(0.85, 1.0); ax.set_ylabel('malignant-call macro-F1')
ax.set_title('A single CNV scalar rivals a 30-dim embedding'); fig.tight_layout()
print({k: round(v, 4) for k, v in abl.items() if k.endswith('mean') or 'lift' in k or 'separation' in k})
